# Baselines: rus-scifact и rus-nfcorpus

Полный прогон бейзлайнов: **me5-large**, **BM25**, **GigaEmbeddings**, **RRF(me5+bm25)**, **RRF(giga+bm25)**.

Метрики: **NDCG@10**, **NDCG@100**, **Recall@10**, **Recall@100**, **MRR@10**.

Расчёт на A100 80GB. Embedding-модели держим в fp16, большие батчи, FAISS IndexFlatIP на GPU.

## 0. Установка зависимостей

In [ ]:
!pip install -q datasets 'transformers==4.51.0' 'sentence-transformers==5.1.1' faiss-gpu-cu12 rank-bm25 pytrec_eval tqdm einops

In [ ]:
import os
import gc
import json
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import pytrec_eval

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem total (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path('./cache')
CACHE_DIR.mkdir(exist_ok=True)

## 1. Загрузка датасетов

В обоих датасетах `corpus` и `queries` лежат в split'е `train`, qrels — отдельный репозиторий. Для scifact используем `test` qrels, для nfcorpus тоже `test`.

In [ ]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    """Загружает корпус, запросы и qrels одного из kaengreg-датасетов."""
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    # corpus: _id -> {'title', 'text', 'processed_title', 'processed_text'}
    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
            'processed_title': row.get('processed_title') or '',
            'processed_text': row.get('processed_text') or '',
        }

    # qrels: query-id -> {corpus-id -> score}
    qrels = defaultdict(dict)
    for row in qrels_ds:
        qid = str(row['query-id'])
        did = str(row['corpus-id'])
        qrels[qid][did] = int(row['score'])

    # Берём только те запросы, для которых есть qrels в этом split'е
    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {
                'text': row.get('text') or '',
                'processed_text': row.get('processed_text') or '',
            }

    # На случай, если в qrels есть qid'ы без текста (отфильтруем)
    qrels = {qid: rels for qid, rels in qrels.items() if qid in queries}

    print(f'{name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)

In [ ]:
scifact_corpus, scifact_queries, scifact_qrels = load_beir_like('rus-scifact', 'test')
nfcorpus_corpus, nfcorpus_queries, nfcorpus_qrels = load_beir_like('rus-nfcorpus', 'test')

## 2. Метрики (pytrec_eval)

Считаем стандартные BEIR-метрики. Возвращаем как средние, так и per-query (последнее пригодится позже для стратификации).

In [ ]:
K_VALUES = [10, 100]

def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    """run: {qid: {docid: score}} — лучше score, выше ранг."""
    metric_strings = set()
    for k in k_values:
        metric_strings.add(f'ndcg_cut.{k}')
        metric_strings.add(f'recall.{k}')
    metric_strings.add('recip_rank')  # MRR

    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strings)
    per_query = evaluator.evaluate(run)

    agg = defaultdict(list)
    for qid, metrics in per_query.items():
        for m, v in metrics.items():
            agg[m].append(v)
    means = {m: float(np.mean(v)) for m, v in agg.items()}

    # Переименуем для наглядности
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
    out['MRR'] = means.get('recip_rank', 0.0)
    return out, per_query

## 3. Dense ретривер: общая утилита

Один кодировщик для всех dense-моделей. Маневрируем батчем под A100 80GB: для base-моделей берём 256, для large — 128.

Тексты документов = `title + ' ' + text` (как в BEIR). Эмбеддинги храним на GPU в fp16, поиск через FAISS GPU IndexFlatIP.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@torch.inference_mode()
def encode_texts(
    texts,
    model,
    tokenizer,
    batch_size: int = 128,
    max_length: int = 512,
    pooling: str = 'mean',  # 'mean' | 'cls'
    normalize: bool = True,
    prefix: str = '',
    desc: str = 'encode',
):
    """Кодирует список текстов; возвращает float32 numpy (N, dim)."""
    model.eval()
    if prefix:
        texts = [prefix + t for t in texts]

    # Сортируем по длине для меньшего паддинга
    lengths = [len(t) for t in texts]
    order = np.argsort(lengths)
    texts_sorted = [texts[i] for i in order]

    all_emb = [None] * len(texts)
    for start in tqdm(range(0, len(texts_sorted), batch_size), desc=desc):
        batch = texts_sorted[start:start + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt'
        ).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(**enc)
        if pooling == 'mean':
            emb = mean_pool(out.last_hidden_state, enc['attention_mask'])
        elif pooling == 'cls':
            emb = out.last_hidden_state[:, 0]
        else:
            raise ValueError(pooling)
        if normalize:
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        emb = emb.float().cpu().numpy()
        for i, idx in enumerate(order[start:start + batch_size]):
            all_emb[idx] = emb[i]
    return np.vstack(all_emb)

In [ ]:
def faiss_search_gpu(doc_emb: np.ndarray, query_emb: np.ndarray, k: int = 100):
    """Точный поиск IP на GPU. doc_emb и query_emb должны быть L2-нормализованы для cosine."""
    dim = doc_emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(doc_emb.astype(np.float32))
    scores, idxs = index.search(query_emb.astype(np.float32), k)
    return scores, idxs


def build_run(qids, doc_ids, scores, idxs):
    """Из выходов FAISS делает run-словарь {qid: {docid: score}}."""
    run = {}
    for i, qid in enumerate(qids):
        run[qid] = {}
        for j, doc_idx in enumerate(idxs[i]):
            if doc_idx < 0:
                continue
            run[qid][doc_ids[doc_idx]] = float(scores[i][j])
    return run

## 4. BM25

Используем `rank_bm25` поверх готовых полей `processed_title` / `processed_text` из датасета — они уже лемматизированы и очищены, отдельный препроцессинг не нужен. На таких корпусах (3.6k–5k документов) скоринг занимает секунды.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize_processed(text: str) -> list:
    """`processed_*` поля уже лемматизированы и очищены — достаточно split."""
    return text.split()


def run_bm25(corpus: dict, queries: dict, k: int = 100, dataset_name: str = ''):
    cache_path = CACHE_DIR / f'bm25_{dataset_name}.json'
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    doc_ids = list(corpus.keys())
    print('Tokenizing corpus...')
    tokenized_corpus = [
        tokenize_processed(
            (corpus[d]['processed_title'] + ' ' + corpus[d]['processed_text']).strip()
        )
        for d in tqdm(doc_ids)
    ]
    bm25 = BM25Okapi(tokenized_corpus)

    qids = list(queries.keys())
    print('Tokenizing queries...')
    tokenized_queries = [tokenize_processed(queries[q]['processed_text']) for q in tqdm(qids)]

    print('Scoring...')
    run = {}
    for qid, q_tokens in zip(tqdm(qids), tokenized_queries):
        scores = bm25.get_scores(q_tokens)
        top_k_idx = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
        top_k_idx = top_k_idx[np.argsort(-scores[top_k_idx])]
        run[qid] = {doc_ids[i]: float(scores[i]) for i in top_k_idx}

    with open(cache_path, 'w') as f:
        json.dump(run, f)
    return run


## 5. Конфиги embedding-моделей и общий dense-прогон

In [ ]:
MODEL_CONFIGS = {
    'me5-large': {
        'hf_id': 'intfloat/multilingual-e5-large',
        'loader': 'hf',
        'pooling': 'mean',
        'dtype': torch.float16,
        'doc_prefix': 'passage: ',
        'query_prefix': 'query: ',
        'batch_size': 128,
        'max_length': 512,
    },
    'giga-embeddings': {
        'hf_id': 'ai-sage/Giga-Embeddings-instruct',
        # giga использует Latent-Attention Pooling (часть архитектуры) — не делаем mean_pool сами,
        # а вызываем модель с return_embeddings=True (см. карточку модели).
        'loader': 'giga',
        'dtype': torch.bfloat16,  # карточка рекомендует bf16
        'doc_prefix': '',
        # Prompt из карточки модели (Russian retrieval):
        'query_prefix': 'Instruct: Дан вопрос, найди подходящий документ который отвечает на вопрос\nQuestion: ',
        'batch_size': 16,  # модель крупнее (~2.5B), держим запас памяти
        'max_length': 4096,
    },
}

In [ ]:
@torch.inference_mode()
def encode_giga(texts, model, tokenizer, batch_size, max_length, prefix, desc):
    """Кодировщик для Giga-Embeddings: использует return_embeddings=True (Latent-Attention Pooling)."""
    if prefix:
        texts = [prefix + t for t in texts]
    lengths = [len(t) for t in texts]
    order = np.argsort(lengths)
    texts_sorted = [texts[i] for i in order]

    all_emb = [None] * len(texts)
    for start in tqdm(range(0, len(texts_sorted), batch_size), desc=desc):
        batch = texts_sorted[start:start + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt'
        ).to(DEVICE)
        emb = model(**enc, return_embeddings=True)
        # На случай, если модель вернула tuple/objект — берём тензор
        if not torch.is_tensor(emb):
            emb = emb[0] if isinstance(emb, (tuple, list)) else emb.embeddings
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        emb = emb.float().cpu().numpy()
        for i, idx in enumerate(order[start:start + batch_size]):
            all_emb[idx] = emb[i]
    return np.vstack(all_emb)


def run_dense(model_name: str, corpus: dict, queries: dict, k: int = 100, dataset_name: str = ''):
    cfg = MODEL_CONFIGS[model_name]
    cache_emb = CACHE_DIR / f'{model_name}_{dataset_name}_doc.npy'
    cache_run = CACHE_DIR / f'{model_name}_{dataset_name}_run.json'
    if cache_run.exists():
        with open(cache_run) as f:
            return json.load(f)

    print(f'Loading {cfg["hf_id"]}')
    tokenizer = AutoTokenizer.from_pretrained(cfg['hf_id'], trust_remote_code=True)

    # Пробуем загрузить с flash-attention 2 (быстрее), откатываемся на eager если flash_attn не установлен
    load_kwargs = dict(trust_remote_code=True, torch_dtype=cfg['dtype'])
    if cfg['loader'] == 'giga':
        try:
            model = AutoModel.from_pretrained(
                cfg['hf_id'], attn_implementation='flash_attention_2', **load_kwargs
            ).to(DEVICE)
        except (ImportError, ValueError) as e:
            print(f'  flash_attention_2 unavailable ({type(e).__name__}); falling back to eager.')
            model = AutoModel.from_pretrained(cfg['hf_id'], **load_kwargs).to(DEVICE)
    else:
        model = AutoModel.from_pretrained(cfg['hf_id'], **load_kwargs).to(DEVICE)
    model.eval()

    doc_ids = list(corpus.keys())
    qids = list(queries.keys())

    if cache_emb.exists():
        doc_emb = np.load(cache_emb)
        print('Loaded cached doc embeddings:', doc_emb.shape)
    else:
        doc_texts = [(corpus[d]['title'] + ' ' + corpus[d]['text']).strip() for d in doc_ids]
        if cfg['loader'] == 'giga':
            doc_emb = encode_giga(
                doc_texts, model, tokenizer,
                batch_size=cfg['batch_size'], max_length=cfg['max_length'],
                prefix=cfg['doc_prefix'], desc=f'{model_name}/docs',
            )
        else:
            doc_emb = encode_texts(
                doc_texts, model, tokenizer,
                batch_size=cfg['batch_size'], max_length=cfg['max_length'],
                pooling=cfg['pooling'], normalize=True,
                prefix=cfg['doc_prefix'], desc=f'{model_name}/docs',
            )
        np.save(cache_emb, doc_emb)

    query_texts = [queries[q]['text'] for q in qids]
    if cfg['loader'] == 'giga':
        query_emb = encode_giga(
            query_texts, model, tokenizer,
            batch_size=cfg['batch_size'], max_length=cfg['max_length'],
            prefix=cfg['query_prefix'], desc=f'{model_name}/queries',
        )
    else:
        query_emb = encode_texts(
            query_texts, model, tokenizer,
            batch_size=cfg['batch_size'], max_length=cfg['max_length'],
            pooling=cfg['pooling'], normalize=True,
            prefix=cfg['query_prefix'], desc=f'{model_name}/queries',
        )

    # Освобождаем модель — embedding'и уже на CPU
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    print('FAISS search...')
    scores, idxs = faiss_search_gpu(doc_emb, query_emb, k=k)
    run = build_run(qids, doc_ids, scores, idxs)

    with open(cache_run, 'w') as f:
        json.dump(run, f)
    return run

## 6. Reciprocal Rank Fusion

Стандартный RRF с `k=60`. Считаем по всем документам, которые вошли хотя бы в один из топов.

In [ ]:
def rrf_fuse(runs: list, k_rrf: int = 60, top_k: int = 100):
    """runs: список run'ов вида {qid: {docid: score}}; берётся ранг внутри каждого."""
    qids = set()
    for r in runs:
        qids.update(r.keys())

    fused = {}
    for qid in qids:
        agg = defaultdict(float)
        for r in runs:
            if qid not in r:
                continue
            ranked = sorted(r[qid].items(), key=lambda x: -x[1])
            for rank, (docid, _) in enumerate(ranked, start=1):
                agg[docid] += 1.0 / (k_rrf + rank)
        top = sorted(agg.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {d: float(s) for d, s in top}
    return fused

## 7. Прогон по обоим датасетам

In [ ]:
DATASETS = {
    'rus-scifact': (scifact_corpus, scifact_queries, scifact_qrels),
    'rus-nfcorpus': (nfcorpus_corpus, nfcorpus_queries, nfcorpus_qrels),
}

all_results = {}  # {dataset: {method: metrics}}
all_runs = {}     # {dataset: {method: run}}

for ds_name, (corpus, queries, qrels) in DATASETS.items():
    print(f'\n{"=" * 60}\nDataset: {ds_name}\n{"=" * 60}')
    runs = {}

    t0 = time.time()
    runs['bm25'] = run_bm25(corpus, queries, k=100, dataset_name=ds_name)
    print(f'BM25 done in {time.time() - t0:.1f}s')

    t0 = time.time()
    runs['me5-large'] = run_dense('me5-large', corpus, queries, k=100, dataset_name=ds_name)
    print(f'me5-large done in {time.time() - t0:.1f}s')

    t0 = time.time()
    runs['giga-embeddings'] = run_dense('giga-embeddings', corpus, queries, k=100, dataset_name=ds_name)
    print(f'giga-embeddings done in {time.time() - t0:.1f}s')

    runs['rrf(me5+bm25)'] = rrf_fuse([runs['me5-large'], runs['bm25']])
    runs['rrf(giga+bm25)'] = rrf_fuse([runs['giga-embeddings'], runs['bm25']])

    metrics = {}
    for method, run in runs.items():
        m, _ = evaluate_run(qrels, run)
        metrics[method] = m
        print(f'  {method:20s}  NDCG@10={m["NDCG@10"]:.4f}  R@100={m["Recall@100"]:.4f}  MRR={m["MRR"]:.4f}')

    all_results[ds_name] = metrics
    all_runs[ds_name] = runs

    with open(RESULTS_DIR / f'{ds_name}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

## 8. Сводная таблица

In [ ]:
import pandas as pd

rows = []
for ds_name, methods in all_results.items():
    for method, m in methods.items():
        rows.append({
            'dataset': ds_name,
            'method': method,
            'NDCG@10': round(m['NDCG@10'], 4),
            'NDCG@100': round(m['NDCG@100'], 4),
            'Recall@10': round(m['Recall@10'], 4),
            'Recall@100': round(m['Recall@100'], 4),
            'MRR': round(m['MRR'], 4),
        })
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / 'baselines_summary.csv', index=False)
df

## 9. Per-query метрики (для следующего шага — стратификации)

Сохраняем NDCG@10 на каждый запрос для лучшего бейзлайна — будем по нему разделять «лёгкие» и «сложные» запросы при отборе сабсета для QE-экспериментов.

In [ ]:
for ds_name, (corpus, queries, qrels) in DATASETS.items():
    runs = all_runs[ds_name]
    # выбираем лучший по NDCG@10 как референс
    best_method = max(all_results[ds_name].items(), key=lambda kv: kv[1]['NDCG@10'])[0]
    print(f'{ds_name}: best baseline = {best_method}')
    _, per_q = evaluate_run(qrels, runs[best_method])
    pq = {qid: {'ndcg@10': m['ndcg_cut_10'], 'recall@100': m['recall_100']} for qid, m in per_q.items()}
    with open(RESULTS_DIR / f'{ds_name}_per_query_{best_method}.json', 'w') as f:
        json.dump(pq, f, indent=2, ensure_ascii=False)